In [17]:
# ===================================================================
# COMPREHENSIVE PROJECT TESTING NOTEBOOK
# Testing all modules: ingestion, validation, processing, analytics, reporting
# ===================================================================

import sys
from pathlib import Path

# Add project root to path
project_root = Path(".").resolve().parent
sys.path.insert(0, str(project_root))

print(f"✓ Project root added to path: {project_root}")
print(f"✓ Python version: {sys.version.split()[0]}")

✓ Project root added to path: E:\PROJECTS_DIR\PY_PROJECTS\python-finsight-project
✓ Python version: 3.11.9


In [18]:
# ===================================================================
# 1. TEST IMPORTS - Verify all modules can be imported
# ===================================================================

print("\n" + "="*60)
print("TEST 1: IMPORTING ALL PROJECT MODULES")
print("="*60)

try:
    # Core modules
    from core.config import ConfigManager
    from core.logger import get_logger
    from core.exceptions import IngestionError, ConfigurationError, ValidationError
    from core.decorators import log_execution, retry
    
    # Ingestion modules
    from ingestion.factory import DataReaderFactory
    from ingestion.api_reader import ApiReader
    from ingestion.csv_reader import CsvReader
    
    # Validation modules
    from validation.validator import DataValidator
    from validation.strategies import NullCheckStrategy, RangeCheckStrategy, OutlierStrategy
    from validation.models import ValidationResult
    
    # Processing modules
    from processing.stock_processor import StockProcessor
    from processing.base_processor import BaseProcessor
    
    # Analytics modules
    from analytics.metrics import PortfolioMetrics
    
    # Reporting modules
    from reporting.alerts import AlertManager, LogAlertObserver
    from reporting.charts import plot_cumulative_returns, plot_correlation_heatmap
    from reporting.dashboard import build_dashboard
    
    # Orchestrator
    from core.orchestrator import PipelineOrchestrator
    
    print("✓ All modules imported successfully!")
    
except Exception as e:
    print(f"✗ Import error: {e}")


TEST 1: IMPORTING ALL PROJECT MODULES
✓ All modules imported successfully!


In [19]:
# ===================================================================
# 2. TEST CONFIG MANAGER - Singleton pattern and config loading
# ===================================================================

print("\n" + "="*60)
print("TEST 2: CONFIG MANAGER (Singleton Pattern)")
print("="*60)

# Test singleton pattern
config1 = ConfigManager.get_instance()
config2 = ConfigManager.get_instance()

print(f"✓ Singleton test: config1 is config2 = {config1 is config2}")

# Load configuration
try:
    # Use absolute path from project root
    config_path = str(project_root / "config" / "settings.yaml")
    config1.load(config_path)
    print("✓ Config loaded successfully")
    
    # Test config retrieval
    project_name = config1.get("project", "name")
    version = config1.get("project", "version")
    start_date = config1.get("pipeline", "start_date")
    end_date = config1.get("pipeline", "end_date")
    
    print(f"  • Project: {project_name} v{version}")
    print(f"  • Date range: {start_date} to {end_date}")
    print(f"  • Price range: {config1.get('validation', 'price_min')} - {config1.get('validation', 'price_max')}")
    
except Exception as e:
    print(f"✗ Config error: {e}")


TEST 2: CONFIG MANAGER (Singleton Pattern)
✓ Singleton test: config1 is config2 = True
✓ Config loaded successfully
  • Project: FinSight v1.0.0
  • Date range: 2022-01-01 to 2024-01-01
  • Price range: 0.01 - 100000.0


In [5]:
# ===================================================================
# 3. TEST DATA INGESTION - Factory pattern and data readers
# ===================================================================

print("\n" + "="*60)
print("TEST 3: DATA INGESTION (Factory Pattern)")
print("="*60)

import pandas as pd
import numpy as np

# Test 1: Factory pattern - create API reader
try:
    print("\n--- Testing API Reader ---")
    reader = DataReaderFactory.create("api")
    print(f"✓ Created reader: {type(reader).__name__}")
    
    # Download sample data for AAPL
    print("  Downloading AAPL data (2023-01-01 to 2023-03-01)...")
    df_api = reader.read(
        ticker="AAPL",
        start="2023-01-01",
        end="2023-03-01"
    )
    
    print(f"✓ Downloaded {len(df_api)} rows")
    print(f"  Shape: {df_api.shape}")
    print(f"  Columns: {list(df_api.columns)}")
    print(f"\n  Sample data:")
    print(df_api.head(3))
    
except Exception as e:
    print(f"✗ API Reader error: {e}")

# Test 2: Factory error handling
try:
    print("\n--- Testing Factory Error Handling ---")
    invalid_reader = DataReaderFactory.create("ftp")
    print("✗ Should have raised IngestionError")
except IngestionError as e:
    print(f"✓ Correctly raised IngestionError: {e}")

{"timestamp": "2026-04-16T07:15:16.684100", "level": "INFO", "module": "decorators", "message": "Starting: read"}



TEST 3: DATA INGESTION (Factory Pattern)

--- Testing API Reader ---
✓ Created reader: ApiReader


{"timestamp": "2026-04-16T07:15:18.437632", "level": "INFO", "module": "decorators", "message": "read produced 39 rows"}
{"timestamp": "2026-04-16T07:15:18.446643", "level": "INFO", "module": "decorators", "message": "Completed: read in 1.76s"}


✓ Downloaded 39 rows
  Shape: (39, 7)
  Columns: [('Date', ''), ('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL'), ('Ticker', '')]

  Sample data:
Price        Date       Close        High         Low        Open     Volume  \
Ticker                   AAPL        AAPL        AAPL        AAPL       AAPL   
0      2023-01-03  123.096024  128.834003  122.210227  128.223793  112117500   
1      2023-01-04  124.365669  126.629372  123.105873  124.887303   89113600   
2      2023-01-05  123.046806  125.753403  122.790915  125.123505   80962700   

Price  Ticker  
Ticker         
0        AAPL  
1        AAPL  
2        AAPL  

--- Testing Factory Error Handling ---
✓ Correctly raised IngestionError: Unknown source type: 'ftp'. Available: ['csv', 'api']


In [6]:
# ===================================================================
# 4. TEST DATA VALIDATION - Multiple validation strategies
# ===================================================================

print("\n" + "="*60)
print("TEST 4: DATA VALIDATION (Strategy Pattern)")
print("="*60)

# Create test data with some issues
test_df = pd.DataFrame({
    'Date': pd.date_range('2023-01-01', periods=100),
    'Ticker': 'TEST',
    'Open': np.random.uniform(100, 150, 100),
    'High': np.random.uniform(150, 160, 100),
    'Low': np.random.uniform(90, 100, 100),
    'Close': np.random.uniform(100, 150, 100),
    'Volume': np.random.randint(1000000, 10000000, 100)
})

# Add some null values
test_df.loc[5:7, 'Close'] = np.nan

# Add some outliers in volume
test_df.loc[10:12, 'Volume'] = 1e9

print(f"Created test dataset: {test_df.shape}")

# Test 1: Null Check Strategy
print("\n--- Null Check Strategy ---")
null_strategy = NullCheckStrategy("Close")
result = null_strategy.validate(test_df)
print(f"✓ Rule: {result.rule_name}")
print(f"  Passed: {result.passed}, Failed: {result.failed}")
print(f"  Quality Score: {result.quality_score}%")

# Test 2: Range Check Strategy
print("\n--- Range Check Strategy ---")
range_strategy = RangeCheckStrategy("Close", min_val=50, max_val=200)
result = range_strategy.validate(test_df)
print(f"✓ Rule: {result.rule_name}")
print(f"  Passed: {result.passed}, Failed: {result.failed}")
print(f"  Quality Score: {result.quality_score}%")

# Test 3: Outlier Detection Strategy
print("\n--- Outlier Detection Strategy ---")
outlier_strategy = OutlierStrategy("Volume", threshold=3.0)
result = outlier_strategy.validate(test_df)
print(f"✓ Rule: {result.rule_name}")
print(f"  Passed: {result.passed}, Failed: {result.failed}")
print(f"  Quality Score: {result.quality_score}%")

# Test 4: Full Validation Pipeline
print("\n--- Full Validation Pipeline ---")
validator = DataValidator([
    NullCheckStrategy("Close"),
    RangeCheckStrategy("Close", min_val=50, max_val=200),
    OutlierStrategy("Volume", threshold=3.0)
])

df_clean, validation_results = validator.validate(test_df.copy())
print(f"✓ Original rows: {len(test_df)}, Clean rows: {len(df_clean)}")
print(f"  Rows removed: {len(test_df) - len(df_clean)}")
print(f"\n  Validation Results:")
for res in validation_results:
    print(f"    • {res.rule_name}: {res.quality_score}%")

{"timestamp": "2026-04-16T07:15:26.461353", "level": "INFO", "module": "decorators", "message": "Starting: validate"}
{"timestamp": "2026-04-16T07:15:26.465385", "level": "INFO", "module": "validator", "message": "Rule 'null_check:Close': score=97.0%"}
{"timestamp": "2026-04-16T07:15:26.471458", "level": "INFO", "module": "validator", "message": "Rule 'range_check:Close[50,200]': score=97.0%"}
{"timestamp": "2026-04-16T07:15:26.473143", "level": "INFO", "module": "validator", "message": "Rule 'outlier_check:Volume(z>3.0)': score=97.0%"}
{"timestamp": "2026-04-16T07:15:26.477158", "level": "INFO", "module": "decorators", "message": "Completed: validate in 0.014s"}



TEST 4: DATA VALIDATION (Strategy Pattern)
Created test dataset: (100, 7)

--- Null Check Strategy ---
✓ Rule: null_check:Close
  Passed: 97, Failed: 3
  Quality Score: 97.0%

--- Range Check Strategy ---
✓ Rule: range_check:Close[50,200]
  Passed: 97, Failed: 3
  Quality Score: 97.0%

--- Outlier Detection Strategy ---
✓ Rule: outlier_check:Volume(z>3.0)
  Passed: 97, Failed: 3
  Quality Score: 97.0%

--- Full Validation Pipeline ---
✓ Original rows: 100, Clean rows: 94
  Rows removed: 6

  Validation Results:
    • null_check:Close: 97.0%
    • range_check:Close[50,200]: 97.0%
    • outlier_check:Volume(z>3.0): 97.0%


In [9]:
# ===================================================================
# 5. TEST DATA PROCESSING - Feature engineering and transformations
# ===================================================================

print("\n" + "="*60)
print("TEST 5: DATA PROCESSING (Feature Engineering)")
print("="*60)

# Create real stock data
processor = StockProcessor()

# Use df_api from earlier (if available) or create test data
try:
    if 'df_api' in locals():
        df_to_process = df_api.copy()
        # Flatten MultiIndex columns if needed
        if isinstance(df_to_process.columns, pd.MultiIndex):
            df_to_process.columns = [col[0] for col in df_to_process.columns.values]
        # Remove MultiIndex from index
        df_to_process = df_to_process.reset_index(drop=True)
    else:
        df_to_process = test_df.copy()
    
    print(f"Columns in data: {list(df_to_process.columns)}")
    print(f"Processing {len(df_to_process)} rows...")
    df_processed = processor.transform(df_to_process)
    
    print(f"✓ Processed data shape: {df_processed.shape}")
    print(f"  New features created:")
    
    new_cols = set(df_processed.columns) - set(df_to_process.columns)
    for col in new_cols:
        print(f"    • {col}")
    
    print(f"\n  Sample processed data:")
    display_cols = ['Close', 'daily_return', 'cumulative_return', 'rolling_vol_20d', 'sma_50', 'sma_200']
    existing_cols = [col for col in display_cols if col in df_processed.columns]
    if existing_cols:
        print(df_processed[existing_cols].head())
    
except Exception as e:
    import traceback
    print(f"✗ Processing error: {e}")
    traceback.print_exc()


TEST 5: DATA PROCESSING (Feature Engineering)
Columns in data: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
Processing 39 rows...
✓ Processed data shape: (38, 13)
  New features created:
    • sma_200
    • rolling_vol_20d
    • log_return
    • cumulative_return
    • daily_return
    • sma_50

  Sample processed data:
        Close  daily_return  cumulative_return  rolling_vol_20d  sma_50  \
0  124.365669      0.010314           0.010314              NaN     NaN   
1  123.046806     -0.010605          -0.000400              NaN     NaN   
2  127.574211      0.036794           0.036380              NaN     NaN   
3  128.095840      0.004089           0.040617              NaN     NaN   
4  128.666687      0.004456           0.045255              NaN     NaN   

   sma_200  
0      NaN  
1      NaN  
2      NaN  
3      NaN  
4      NaN  


In [10]:
# ===================================================================
# 6. TEST ANALYTICS - Portfolio metrics calculation
# ===================================================================

print("\n" + "="*60)
print("TEST 6: ANALYTICS (Portfolio Metrics)")
print("="*60)

try:
    if 'df_processed' in locals() and len(df_processed) > 0:
        # Extract returns for metrics calculation
        returns = df_processed['daily_return'].dropna().values
        cum_returns = df_processed['cumulative_return'].values
        
        # Calculate metrics
        print("\n--- Calculating Portfolio Metrics ---")
        
        sharpe = PortfolioMetrics.sharpe_ratio(returns)
        print(f"✓ Sharpe Ratio: {sharpe:.4f}")
        
        max_dd = PortfolioMetrics.max_drawdown(cum_returns)
        print(f"✓ Max Drawdown: {max_dd:.4f}")
        
        var_95 = PortfolioMetrics.value_at_risk(returns, confidence=0.95)
        print(f"✓ Value at Risk (95%): {var_95:.4f}")
        
        volatility = PortfolioMetrics.annualised_volatility(returns)
        print(f"✓ Annualised Volatility: {volatility:.4f}")
        
        # Interpretation
        print("\n--- Metrics Interpretation ---")
        print(f"  • Sharpe Ratio > 0.5 is generally considered good")
        print(f"  • Max Drawdown shows worst peak-to-trough decline")
        print(f"  • VaR indicates potential loss at 95% confidence")
        print(f"  • Volatility shows annualized price swings")
        
except Exception as e:
    print(f"✗ Analytics error: {e}")


TEST 6: ANALYTICS (Portfolio Metrics)

--- Calculating Portfolio Metrics ---
✓ Sharpe Ratio: 4.6214
✓ Max Drawdown: -0.0555
✓ Value at Risk (95%): -0.0183
✓ Annualised Volatility: 0.2359

--- Metrics Interpretation ---
  • Sharpe Ratio > 0.5 is generally considered good
  • Max Drawdown shows worst peak-to-trough decline
  • VaR indicates potential loss at 95% confidence
  • Volatility shows annualized price swings


In [11]:
# ===================================================================
# 7. TEST ALERTS - Alert system with observers
# ===================================================================

print("\n" + "="*60)
print("TEST 7: ALERT SYSTEM (Observer Pattern)")
print("="*60)

try:
    # Create alert manager
    alert_manager = AlertManager()
    alert_manager.subscribe(LogAlertObserver())
    
    print("✓ AlertManager created with LogAlertObserver")
    
    # Test alert checks
    print("\n--- Testing Alert Thresholds ---")
    
    # Check 1: Sharpe ratio (should trigger - below threshold)
    alert_manager.check(
        metric="sharpe_ratio",
        value=0.3,  # Below threshold of 0.5
        threshold=0.5,
        condition="below"
    )
    print("✓ Alert check for Sharpe ratio executed")
    
    # Check 2: Sharpe ratio (should NOT trigger - above threshold)
    alert_manager.check(
        metric="sharpe_ratio",
        value=0.8,  # Above threshold of 0.5
        threshold=0.5,
        condition="below"
    )
    print("✓ Alert check for high Sharpe ratio executed (no alert)")
    
except Exception as e:
    print(f"✗ Alert error: {e}")

{"timestamp": "2026-04-16T07:16:29.463192", "level": "WARNING", "module": "alerts", "message": "ALERT: sharpe_ratio = 0.3000 breached threshold 0.5"}



TEST 7: ALERT SYSTEM (Observer Pattern)
✓ AlertManager created with LogAlertObserver

--- Testing Alert Thresholds ---
✓ Alert check for Sharpe ratio executed
✓ Alert check for high Sharpe ratio executed (no alert)


In [12]:
# ===================================================================
# 8. TEST DECORATORS - Logging and retry mechanisms
# ===================================================================

print("\n" + "="*60)
print("TEST 8: DECORATORS (Logging & Retry)")
print("="*60)

# Define test function with decorators
@log_execution
def test_operation(data_size: int):
    """Test function with logging decorator"""
    result = sum(range(data_size))
    return result

@retry(max_attempts=3)
def test_retry_operation(attempt_counter=[0]):
    """Test function that retries on failure"""
    attempt_counter[0] += 1
    if attempt_counter[0] < 2:
        raise ValueError(f"Attempt {attempt_counter[0]} failed")
    return f"Success on attempt {attempt_counter[0]}"

try:
    print("\n--- Testing @log_execution Decorator ---")
    result = test_operation(1000)
    print(f"✓ Operation completed with result: {result}")
    
except Exception as e:
    print(f"✗ Decorator error: {e}")

try:
    print("\n--- Testing @retry Decorator ---")
    attempt_counter = [0]
    
    @retry(max_attempts=3)
    def retry_test(counter):
        counter[0] += 1
        if counter[0] < 2:
            raise ValueError(f"Attempt {counter[0]} failed")
        return f"Success on attempt {counter[0]}"
    
    result = retry_test(attempt_counter)
    print(f"✓ Retry test passed: {result}")
    
except Exception as e:
    print(f"✗ Retry error: {e}")

{"timestamp": "2026-04-16T07:16:34.111397", "level": "INFO", "module": "decorators", "message": "Starting: test_operation"}
{"timestamp": "2026-04-16T07:16:34.113761", "level": "INFO", "module": "decorators", "message": "Completed: test_operation in 0.0s"}
{"timestamp": "2026-04-16T07:16:34.115312", "level": "WARNING", "module": "decorators", "message": "retry_test failed (attempt 1): Attempt 1 failed. Retrying..."}



TEST 8: DECORATORS (Logging & Retry)

--- Testing @log_execution Decorator ---
✓ Operation completed with result: 499500

--- Testing @retry Decorator ---
✓ Retry test passed: Success on attempt 2


In [15]:
# ===================================================================
# 9. TEST COMPLETE PIPELINE - Full orchestration
# ===================================================================

print("\n" + "="*60)
print("TEST 9: COMPLETE PIPELINE (Manual Orchestration)")
print("="*60)

try:
    print("✓ Pipeline components initialized")
    print("\n--- Running Complete Pipeline for AAPL ---")
    
    # Step 1: Ingestion
    print("\nStep 1: Data Ingestion")
    reader = DataReaderFactory.create("api")
    df_raw = reader.read(
        ticker="AAPL",
        start="2023-01-01",
        end="2023-03-01"
    )
    print(f"  ✓ Downloaded {len(df_raw)} rows")
    
    # Flatten columns
    if isinstance(df_raw.columns, pd.MultiIndex):
        df_raw.columns = [col[0] for col in df_raw.columns.values]
    df_raw = df_raw.reset_index(drop=True)
    
    # Step 2: Validation
    print("\nStep 2: Data Validation")
    validator = DataValidator([
        NullCheckStrategy("Close"),
        RangeCheckStrategy("Close", 0.01, 100000.0),
        OutlierStrategy("Volume", 3.0)
    ])
    df_clean, val_results = validator.validate(df_raw)
    print(f"  ✓ Validated {len(df_clean)} rows (removed {len(df_raw) - len(df_clean)})")
    
    # Step 3: Processing
    print("\nStep 3: Feature Engineering")
    processor = StockProcessor()
    df_processed = processor.transform(df_clean)
    print(f"  ✓ Created {len(df_processed.columns)} columns")
    
    # Step 4: Analytics
    print("\nStep 4: Metrics Calculation")
    returns = df_processed['daily_return'].dropna().values
    cum_returns = df_processed['cumulative_return'].values
    
    metrics = {
        "sharpe_ratio": PortfolioMetrics.sharpe_ratio(returns),
        "max_drawdown": PortfolioMetrics.max_drawdown(cum_returns),
        "var_95": PortfolioMetrics.value_at_risk(returns),
        "annualised_vol": PortfolioMetrics.annualised_volatility(returns),
    }
    
    print("\n✓ Pipeline completed successfully!")
    print(f"\n  Final Metrics:")
    for metric_name, metric_value in metrics.items():
        print(f"    • {metric_name}: {metric_value:.6f}")
    
    print(f"\n  Processed Data Shape: {df_processed.shape}")
    if 'Date' in df_processed.columns:
        print(f"  Date Range: {df_processed['Date'].min()} to {df_processed['Date'].max()}")
    
except Exception as e:
    import traceback
    print(f"✗ Pipeline error: {e}")
    print(f"\nTraceback:")
    traceback.print_exc()

{"timestamp": "2026-04-16T07:17:11.835309", "level": "INFO", "module": "decorators", "message": "Starting: read"}


{"timestamp": "2026-04-16T07:17:11.918341", "level": "INFO", "module": "decorators", "message": "read produced 39 rows"}
{"timestamp": "2026-04-16T07:17:11.918341", "level": "INFO", "module": "decorators", "message": "Completed: read in 0.079s"}
{"timestamp": "2026-04-16T07:17:11.924348", "level": "INFO", "module": "decorators", "message": "Starting: validate"}
{"timestamp": "2026-04-16T07:17:11.928597", "level": "INFO", "module": "validator", "message": "Rule 'null_check:Close': score=100.0%"}
{"timestamp": "2026-04-16T07:17:11.935929", "level": "INFO", "module": "validator", "message": "Rule 'range_check:Close[0.01,100000.0]': score=100.0%"}
{"timestamp": "2026-04-16T07:17:11.940868", "level": "INFO", "module": "validator", "message": "Rule 'outlier_check:Volume(z>3.0)': score=97.44%"}
{"timestamp": "2026-04-16T07:17:11.948750", "level": "INFO", "module": "decorators", "message": "Completed: validate in 0.024s"}



TEST 9: COMPLETE PIPELINE (Manual Orchestration)
✓ Pipeline components initialized

--- Running Complete Pipeline for AAPL ---

Step 1: Data Ingestion
  ✓ Downloaded 39 rows

Step 2: Data Validation
  ✓ Validated 38 rows (removed 1)

Step 3: Feature Engineering
  ✓ Created 13 columns

Step 4: Metrics Calculation

✓ Pipeline completed successfully!

  Final Metrics:
    • sharpe_ratio: 4.947463
    • max_drawdown: -0.055495
    • var_95: -0.018420
    • annualised_vol: 0.225899

  Processed Data Shape: (37, 13)
  Date Range: 2023-01-04 00:00:00 to 2023-02-28 00:00:00


In [16]:
# ===================================================================
# 10. TEST SUMMARY - Overall test results
# ===================================================================

print("\n" + "="*60)
print("TEST SUMMARY")
print("="*60)

summary = """
✓ Module 1: Imports - All modules successfully imported
✓ Module 2: Config Manager - Singleton pattern working, config loaded
✓ Module 3: Data Ingestion - Factory pattern, API reader, data fetching
✓ Module 4: Data Validation - Multiple strategies (Null, Range, Outlier)
✓ Module 5: Data Processing - Feature engineering and transformations
✓ Module 6: Analytics - Portfolio metrics calculation
✓ Module 7: Alert System - Observer pattern with alerts
✓ Module 8: Decorators - Logging and retry mechanisms
✓ Module 9: Complete Pipeline - Full orchestration workflow

KEY FEATURES TESTED:
• Design Patterns: Singleton, Factory, Strategy, Observer, Decorator
• Data Pipeline: Ingestion → Validation → Processing → Analytics
• Error Handling: Custom exceptions, retry logic
• Configuration Management: YAML-based config with singleton
• Portfolio Analysis: Sharpe ratio, drawdown, VaR, volatility
• Alert System: Threshold monitoring with observer pattern
• Feature Engineering: Returns, moving averages, volatility metrics

NEXT STEPS:
1. Run individual cells to test specific functionality
2. Modify test parameters as needed
3. Add custom test cases for edge cases
4. Integrate with production data sources
"""

print(summary)


TEST SUMMARY

✓ Module 1: Imports - All modules successfully imported
✓ Module 2: Config Manager - Singleton pattern working, config loaded
✓ Module 3: Data Ingestion - Factory pattern, API reader, data fetching
✓ Module 4: Data Validation - Multiple strategies (Null, Range, Outlier)
✓ Module 5: Data Processing - Feature engineering and transformations
✓ Module 6: Analytics - Portfolio metrics calculation
✓ Module 7: Alert System - Observer pattern with alerts
✓ Module 8: Decorators - Logging and retry mechanisms
✓ Module 9: Complete Pipeline - Full orchestration workflow

KEY FEATURES TESTED:
• Design Patterns: Singleton, Factory, Strategy, Observer, Decorator
• Data Pipeline: Ingestion → Validation → Processing → Analytics
• Error Handling: Custom exceptions, retry logic
• Configuration Management: YAML-based config with singleton
• Portfolio Analysis: Sharpe ratio, drawdown, VaR, volatility
• Alert System: Threshold monitoring with observer pattern
• Feature Engineering: Returns, m